# 2. Create Dataset Using MAPS

In [ ]:
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

from maps.dataset import Dataset
from maps.parameters import ParameterSpace, ParameterSweep

In [ ]:
dataset = Dataset(root='/work/dldevel/galella/datasets/MAPS/scenes')

In [ ]:
scene = dataset.get_scene('002_white-shark')
print(scene)

In [ ]:
params = scene.get()

for param, value in params.items():
    print(f'{param}: {value}')

In [ ]:
dict_space = {
    'camera.azimuth': {'circular': True, 'range': [0, 2 * np.pi]},
    'camera.distance': {'range': [2, 10]},
    'camera.elevation': {'range': [-np.pi / 2, np.pi / 2]},
}

parameter_space = ParameterSpace(dict_space)
print(parameter_space)

In [ ]:
dict_sweep = {
    'camera.azimuth': 20,
    'camera.distance': 5,
    'camera.elevation': 5,
}

In [ ]:
parameter_sweep = ParameterSweep(dict_space, dict_sweep)

In [ ]:
for params in parameter_sweep:
    print(params)

In [ ]:
path_dataset = Path().resolve() / Path('dataset') / 'white_shark_camera'
path_dataset.mkdir(exist_ok=True, parents=True)

path_images = path_dataset / Path('images')
path_images.mkdir(exist_ok=True, parents=True)

path_csv = path_dataset / Path('parameters.csv')

In [ ]:
list_images_names = []

for i, params in tqdm(enumerate(parameter_sweep), total=parameter_sweep.num_images):
    path_image = path_images / f'{i:04d}.png'
    list_images_names.append(path_image.name)
    scene.set(params)
    scene.render(path_image)

In [ ]:
df_sweep = parameter_sweep.to_dataframe()
assert len(df_sweep) == len(list_images_names)

df_sweep.insert(0, 'image', list_images_names)
df_sweep.to_csv(path_csv, index=False)